# Fine-Tuning-Style Baseline And Limits

This notebook uses provider calls with few-shot style examples, but no retrieved policy context. That makes the behavior-adaptation effect visible without running a real fine-tuning job.


## Learning Objectives

By the end of this notebook, you should be able to:

- Explain what supervised and parameter-efficient fine-tuning change.
- Use few-shot examples as a proxy for behavior adaptation.
- Separate tone imitation from the need for policy evidence.
- Explain the limits of the few-shot proxy.


## Fine-Tuning Recap

**Supervised fine-tuning** teaches a model from examples of desired inputs and outputs. **Parameter-efficient fine-tuning** (PEFT) adds or updates a small set of parameters instead of changing every base-model weight. Full fine-tuning keeps a separate copy of the model for each task. Prefix tuning freezes the pretrained model and trains only a small task-specific prefix. This is the same idea as the LoRA adapters from the previous unit: the new behavior is stored in a small set of trained parameters.

<div align="center">
  <img src="assets/04-prefix-tuning.png" alt="Full fine-tuning compared with prefix tuning" width="550" style="max-width: 100%; height: auto;" />
  <p><em>Full fine-tuning (one model copy per task) compared with prefix tuning (a small trained prefix on a frozen model). Source: <a href="https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/prefix-tuning.png" target="_blank" rel="noopener noreferrer">Hugging Face PEFT documentation</a></em></p>
</div>

These approaches can improve repeated behavior, but they need careful evaluation because the model may also learn unwanted shortcuts.

## Few-Shot As A Proxy

Instead of training, we put the style examples in the prompt. This is a low-cost probe: we can see whether examples help tone, refusal style, or answer shape before deciding that real fine-tuning is worth the cost. If examples do not help even in the prompt, fine-tuning is unlikely to be the first thing to try. If they help but the behavior is still inconsistent across held-out cases, a fine-tuning run may be worth scoping with the workflow from the previous unit.

## Limitations

Few-shot examples are sent with one request and disappear afterward. They do not prove that a trained adapter would generalize, preserve safety behavior, or avoid regressions on unrelated tasks. Real fine-tuning still needs a held-out evaluation set and a rollback plan.

## Load Shared Data And Runtime Helpers


In [ ]:
import json
from pprint import pprint
from typing import Literal

from pydantic import BaseModel, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    ReplySource,
    call_model,
    compact_style_examples,
    style_messages,
)

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | Model: {CONFIG.model} | Dry run: {CONFIG.dry_run}"
)

## Inspect Few-Shot Style Messages

`style_messages` turns the four style examples into alternating user and assistant messages, and every question in this notebook is sent after them. The compact view below also shows each example's intent and tone labels. The model sees the user and assistant text, and the system prompt below lists the allowed tone labels, but the model is not told which label belongs to which example.

In [ ]:
few_shot_messages = style_messages(STYLE_EXAMPLES)
print(f"Few-shot message count: {len(few_shot_messages)}")
print(compact_style_examples(STYLE_EXAMPLES))

## Call The Model Without Fresh Policy Context

The model receives style examples, but not policy documents. This intentionally tests the limitation: behavior can improve while factual grounding remains unproven.


The schema separates tone imitation from evidence needs: `predicted_tone` labels the response style with one of the four tones from the data, and `needs_policy_evidence` asks the model whether it could answer without the policy. That field is not a perfect judge, but it makes the limitation visible.

In [ ]:
class StyleOnlyAnswer(BaseModel):
    """Structured output for style-conditioned answers without retrieval."""

    answer: str
    predicted_tone: Literal[
        "concise", "operational", "careful_refusal", "coaching", "unknown"
    ]
    needs_policy_evidence: bool
    response_source: ReplySource = "simulated"

Build the style-only prompt in a helper. Notice that no policy context is supplied.

In [ ]:
def style_only_messages(question: str) -> list[dict[str, str]]:
    """Create a few-shot style prompt without retrieved evidence."""
    return [
        {
            "role": "system",
            "content": (
                "Use the style of the examples. Return JSON with answer, predicted_tone, and needs_policy_evidence. "
                "predicted_tone must be one of: concise, operational, careful_refusal, coaching."
            ),
        },
        *few_shot_messages,
        {"role": "user", "content": question},
    ]

The dry output intentionally says policy evidence is still needed.


In [ ]:
dry_style_only = json.dumps(
    {
        "answer": "Please send the invoice number and both payment references. I need to check the current refund policy before confirming eligibility.",
        "predicted_tone": "concise",
        "needs_policy_evidence": True,
    }
)

Validate the structured style-only response before using it in comparisons.


In [ ]:
def style_only_answer(question: str) -> StyleOnlyAnswer:
    """Ask the provider to answer from style examples without policy context."""
    reply = call_model(
        style_only_messages(question),
        dry_response=dry_style_only,
        response_format={"type": "json_object"},
    )

    try:
        result = StyleOnlyAnswer.model_validate_json(reply.text)
        result.response_source = reply.source
        return result
    except ValidationError:
        return StyleOnlyAnswer(
            answer=reply.text,
            predicted_tone="unknown",
            needs_policy_evidence=True,
            response_source=reply.source,
        )

## Reading Style-Only Outputs

We read each answer with two questions in mind:

- Did it follow the expected behavior pattern: concise, operational, careful refusal, or coaching?
- Did it claim policy facts that were not supplied in the prompt?

A style-only answer can succeed when the task is mostly about response behavior, such as asking for a missing blocker in a coaching tone. It fails when the task needs current evidence: asked whether a duplicate payment can be refunded, the model may imitate the refund-response style but invent eligibility rules without the current policy.

In [ ]:
style_only_outputs = {
    request["request_id"]: style_only_answer(request["question"])
    for request in REQUESTS
}

for request in REQUESTS[:3]:
    print()
    print(request["request_id"], request["question"])
    pprint(style_only_outputs[request["request_id"]].model_dump())

Compare these answers with the policy documents, which were not supplied as context. Some policy facts were still supplied through the style examples, as notebook 02 showed. In live runs, style-only answers often follow the requested tone but fill the gaps from general knowledge: extra refund requirements such as payment dates, amounts, or proof of payment, and specific retry delays or retry limits. None of these come from our policies. The answers sound plausible because they imitate a support style, not because they are grounded.

Also note that `predicted_tone` is the model's own label, not a measurement. An answer can label itself `concise` while listing five requirements.

## Summarize Tone And Evidence Needs

A style-conditioned call can imitate the response pattern, but it still cannot prove the current refund window, rate limit, or MFA rule unless those facts are present in the prompt or already reliably known. The summary below checks whether the model recognizes that.


In [ ]:
# Summarize tone and evidence needs across the held-out requests.
summary_rows = []
for request in REQUESTS:
    output = style_only_outputs[request["request_id"]]
    summary_rows.append(
        {
            "request_id": request["request_id"],
            "expected_tone": request["preferred_tone"],
            "predicted_tone": output.predicted_tone,
            "tone_match": output.predicted_tone == request["preferred_tone"],
            "response_source": output.response_source,
            "needs_policy_evidence": output.needs_policy_evidence,
        }
    )
pprint(summary_rows)

### Exercise: Find Unjustified Confidence

Every request in `REQUESTS` has `fresh_fact_required` set to `True`, so none of them can be answered reliably without policy evidence. Using `summary_rows`, list the request IDs where the style-only model nevertheless reported `needs_policy_evidence` as `False`.


In [ ]:
# @TODO: Find the fresh-fact requests where the model claimed it needed no policy evidence.


<details>
<summary>Show Reference Solution</summary>

```python
fresh_fact_ids = {
    request["request_id"] for request in REQUESTS if request["fresh_fact_required"]
}
overconfident_ids = [
    row["request_id"]
    for row in summary_rows
    if row["request_id"] in fresh_fact_ids and not row["needs_policy_evidence"]
]
overconfident_ids
```

</details>


In live runs, the model often reports that it needs no policy evidence for questions that depend on current policy, and which requests it flags changes from run to run. Its self-assessment is not a reliable grounding check: a model without the policy cannot know what the policy says. In dry mode, and for fallback rows, the deterministic response always says evidence is needed, so the list is empty. Run the notebook with an API key to see the model's own judgment.


## Check Your Understanding

1. Do few-shot examples persist as model behavior after the request is finished?
2. Do LoRA or prefix tuning automatically provide fresh policy evidence at answer time?
3. A style-only answer sounds confident and follows the expected tone. Why can it still not be trusted for policy facts?

<details>
<summary>Show Reference Answers</summary>

1. No. Few-shot examples condition only the current request; they do not update model weights or adapters.
2. No. They can adapt model behavior, but fresh evidence still requires retrieval or another external context source.
3. Style examples teach response behavior but can also repeat policy facts embedded in them. Without current policy evidence, the model cannot verify those facts and may fill gaps with plausible details absent from the policy.

</details>


## Further Reading

- [Does Fine-Tuning LLMs on New Knowledge Encourage Hallucinations?](https://arxiv.org/abs/2405.05904) - Gekhman et al., 2024
- [Few-Shot Parameter-Efficient Fine-Tuning is Better and Cheaper than In-Context Learning](https://arxiv.org/abs/2205.05638) - Liu et al., 2022
